In [1]:
import pandas as pd
import numpy as np

market = pd.read_csv("../data/market_master.csv")

market["Date"] = pd.to_datetime(market["Date"])
market = market.set_index("Date")

In [3]:
#Threshold-Based Signal
threshold = 0.2

market["signal"] = np.where(
    market["mom_20_norm"] > threshold, 1,
    np.where(market["mom_20_norm"] < -threshold, -1, 0)
)

#Persistence Filter
market["signal_smooth"] = market["signal"].rolling(3, min_periods=1).mean()

market["final_signal"] = np.where(
    market["signal_smooth"] > 0.5, 1,
    np.where(market["signal_smooth"] < -0.5, -1, 0)
)

#Position Logic
market["position"] = market["final_signal"].replace(0, np.nan).ffill().fillna(0)

#Trade Detection
market["position_change"] = market["position"].diff().abs().fillna(0)

#Cost Model
cost_per_trade = 0.0005
market["cost"] = market["position_change"] * cost_per_trade

#Strategy Returns
market["strategy_ret"] = (
    market["position"].shift(1) * market["nifty_ret"]
    - market["cost"]
)

market = market.dropna(subset=["strategy_ret"])

In [5]:
market.to_csv("../backtesting/market_backtest_v2.csv")